In [8]:
import cv2
import joblib
import numpy as np

In [9]:
svm = joblib.load("models/svm_face.pkl")
pca = joblib.load("models/pca.pkl")
scaler = joblib.load("models/scaler.pkl")
label_map = joblib.load("models/labels.pkl")

print("✅ Model loaded!")

✅ Model loaded!


In [10]:
def predict_face(face_img):
    face_img = cv2.resize(face_img, (64, 64))
    face_img = face_img.flatten().reshape(1, -1)

    face_img = scaler.transform(face_img)
    face_img = pca.transform(face_img)

    probs = svm.predict_proba(face_img)[0]
    max_prob = np.max(probs)

    if max_prob < 0.6:
        return "Unknown"

    pred = np.argmax(probs)
    return label_map[pred]

In [11]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

cap = cv2.VideoCapture(0)

print("Press Q to exit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]

        name = predict_face(face)

        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, name, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

    cv2.imshow("Face Recognition - SVM", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Press Q to exit
